# Notebook 05 — Modellvergleich: RF · SVM · kNN · MLP

Alle vier Modelle werden auf **identischen Daten** (k=5 ME, 25-s-Fenster) mit **LOO-CV** verglichen.

| Modell | Skalierung | Besonderheit |
|--------|-----------|-------------|
| Random Forest | nein | Baseline (NB04) |
| SVM (RBF) | StandardScaler | Kernel-Methode |
| kNN (k=7) | StandardScaler | Distanzbasiert |
| MLP | StandardScaler | Integer-Labels nötig |

**Bonus (Abschnitt 11):** MLP mit kaskadierten Stage-1-Features — P(Still)/P(Essen) als zusätzliche Stage-2-Features.

---
## 1. Setup

In [ ]:
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

# ── Klassen ───────────────────────────────────────────────────────────────────
CLASSES_RAW    = ['Apfel', 'Kaugummi', 'Skyr', 'Still', 'Essen']
CLASSES_COARSE = ['Still', 'Essen']
CLASSES_FINE   = ['Apfel', 'Kaugummi', 'Skyr', 'Essen']
CLASSES_FULL   = ['Still', 'Apfel', 'Kaugummi', 'Skyr', 'Essen']
TO_COARSE      = {'Apfel': 'Essen', 'Kaugummi': 'Essen',
                  'Skyr':  'Essen', 'Still':    'Still', 'Essen': 'Essen'}
COLORS = {'Apfel': '#e15759', 'Kaugummi': '#4e79a7', 'Skyr': '#f28e2b',
          'Still': '#59a14f', 'Essen': '#b07aa1'}

# ── Daten-Parameter ───────────────────────────────────────────────────────────
FS            = 50.0
TRIM_SECS     = 2
WINDOW_SECS   = 25.0
MIN_TAIL_SECS = 20.0
K             = 5

# ── Modell-Definitionen ───────────────────────────────────────────────────────
# scale=True  → StandardScaler pro LOO-Fold
# le=True     → LabelEncoder (Integer-Labels, nötig für MLP)
MODELS = {
    'RF':  dict(
        clf=lambda: RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
        scale=False, le=False),
    'SVM': dict(
        clf=lambda: SVC(kernel='rbf', C=10, probability=True, class_weight='balanced', random_state=42),
        scale=True, le=False),
    'kNN': dict(
        clf=lambda: KNeighborsClassifier(n_neighbors=7, weights='distance', metric='euclidean'),
        scale=True, le=False),
    'MLP': dict(
        clf=lambda: MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                                   solver='adam', max_iter=500,
                                   learning_rate_init=0.001, random_state=42),
        scale=True, le=True),
}
MODEL_COLORS = {'RF': '#4e79a7', 'SVM': '#e15759', 'kNN': '#f28e2b', 'MLP': '#59a14f'}

# NB04-Referenz
NB04 = {'S1': 0.996, 'S2': 0.900, 'E2E': 0.930}

---
## 2. Daten laden

In [ ]:
DATA_DIR = Path('../data/raw')
_SKIP    = {'Metadata.csv', 'Annotation.csv'}

sessions = {cls: [] for cls in CLASSES_RAW}
for zf in sorted(DATA_DIR.glob('*.zip')):
    for cls in CLASSES_RAW:
        if zf.name.startswith(cls + '_'):
            sessions[cls].append(zf)
            break

print('Sitzungen pro Klasse:')
for cls in CLASSES_RAW:
    print(f'  {cls:12s}: {len(sessions[cls])}')

In [ ]:
def preprocess(df):
    df = df.copy()
    t  = df['seconds_elapsed']
    df = df[(t >= t.iloc[0] + TRIM_SECS) &
            (t <= t.iloc[-1] - TRIM_SECS)].reset_index(drop=True)
    df['lin_x']     = df['accelerationX']
    df['lin_y']     = df['accelerationY']
    df['lin_z']     = df['accelerationZ']
    df['magnitude'] = np.sqrt(df['lin_x']**2 + df['lin_y']**2 + df['lin_z']**2)
    return df

preprocessed = {cls: [] for cls in CLASSES_RAW}
for cls in CLASSES_RAW:
    for zf in sessions[cls]:
        with zipfile.ZipFile(zf) as z:
            csv_name = next(f for f in z.namelist()
                            if f.endswith('.csv') and f not in _SKIP)
            with z.open(csv_name) as f:
                preprocessed[cls].append(preprocess(pd.read_csv(f)))

print(f'Trim: ±{TRIM_SECS} s')

---
## 3. Segmentierung & Movement Exclusion

In [ ]:
def split_windows(df):
    t = df['seconds_elapsed'].values
    t_start, t_end = t[0], t[-1]
    windows = []
    while t_start + MIN_TAIL_SECS <= t_end:
        t_stop = t_start + WINDOW_SECS
        w = df[(t >= t_start) & (t < t_stop)].reset_index(drop=True)
        if len(w) > 1 and (w['seconds_elapsed'].iloc[-1] - w['seconds_elapsed'].iloc[0]) >= MIN_TAIL_SECS:
            windows.append(w)
        t_start = t_stop
    return windows

def movement_mask(df, k=K):
    threshold = max(0.02, k * df['magnitude'].median())
    return df['magnitude'].rolling(50, center=True, min_periods=1).max() <= threshold

windows = {cls: [] for cls in CLASSES_RAW}
for cls in CLASSES_RAW:
    for df in preprocessed[cls]:
        for w in split_windows(df):
            clean = w[movement_mask(w)].reset_index(drop=True)
            if len(clean) > 50:
                windows[cls].append(clean)

print(f'Fenster nach ME (k={K}):')
for cls in CLASSES_RAW:
    print(f'  {cls:12s}: {len(windows[cls]):4d}')
print(f'  {"Gesamt":12s}: {sum(len(v) for v in windows.values()):4d}')

---
## 4. Feature-Extraktion

In [ ]:
def extract_features(df):
    feats = {}
    for col in ['lin_x', 'lin_y', 'lin_z', 'magnitude']:
        feats[f'{col}_mean'] = df[col].mean()
        feats[f'{col}_std']  = df[col].std()
        feats[f'{col}_max']  = df[col].abs().max()
    feats['stillness_ratio'] = (df['magnitude'] < 0.02).mean()
    feats['movement_events'] = int((df['magnitude'] > df['magnitude'].quantile(0.75)).sum())
    for col in ['rotationRateX', 'rotationRateY', 'rotationRateZ']:
        feats[f'{col}_mean'] = df[col].mean()
        feats[f'{col}_std']  = df[col].std()
        feats[f'{col}_max']  = df[col].abs().max()
    for col in ['pitch', 'roll', 'yaw']:
        feats[f'{col}_mean']  = df[col].mean()
        feats[f'{col}_std']   = df[col].std()
        feats[f'{col}_range'] = df[col].max() - df[col].min()
    nperseg = min(256, len(df) // 2)
    freqs, psd = welch(df['magnitude'].values, fs=FS, nperseg=nperseg)
    chew = (freqs >= 0.5) & (freqs <= 4.0)
    cf, cp = freqs[chew], psd[chew]
    feats['total_power']        = float(psd.sum())
    feats['chew_band_power']    = float(cp.sum())
    feats['rhythmicity']        = feats['chew_band_power'] / feats['total_power'] if feats['total_power'] > 0 else 0.0
    feats['dominant_chew_freq'] = float(cf[np.argmax(cp)]) if len(cp) > 0 else 0.0
    return feats

rows, y_raw = [], []
for cls in CLASSES_RAW:
    for df in windows[cls]:
        rows.append(extract_features(df))
        y_raw.append(cls)

X_all    = pd.DataFrame(rows)
y_raw    = np.array(y_raw)
y_coarse = np.array([TO_COARSE[c] for c in y_raw])
eat_mask = y_coarse == 'Essen'
X_eat    = X_all[eat_mask].reset_index(drop=True)
y_eat    = y_raw[eat_mask]

print(f'Gesamt: {len(X_all)} Samples, {X_all.shape[1]} Features')
print(f'  Still: {(~eat_mask).sum()}  |  Essen: {eat_mask.sum()}')
for cls in CLASSES_FINE:
    print(f'    {cls:10s}: {(y_raw==cls).sum()}')

---
## 5. LOO-Hilfsfunktion

In [ ]:
def loo_eval(X_np, y_str, model_cfg):
    """
    Generisches LOO-CV für alle Modelltypen.
    model_cfg: dict mit 'clf' (factory), 'scale' (bool), 'le' (bool)
    Gibt (yt, yp) als String-Arrays zurück.
    """
    le = LabelEncoder().fit(y_str) if model_cfg['le'] else None
    y_fit = le.transform(y_str) if le else y_str
    yt, yp = [], []
    for tr, te in LeaveOneOut().split(X_np):
        Xtr, Xte = X_np[tr], X_np[te]
        if model_cfg['scale']:
            sc  = StandardScaler()
            Xtr = sc.fit_transform(Xtr)
            Xte = sc.transform(Xte)
        clf = model_cfg['clf']()
        clf.fit(Xtr, y_fit[tr])
        pred = clf.predict(Xte)[0]
        if le is not None:
            pred = le.inverse_transform([pred])[0]
        yp.append(pred)
        yt.append(y_str[te][0])
    return np.array(yt), np.array(yp)

print('loo_eval() bereit.')

---
## 6. Stufe 1 — Still vs. Essen  (alle Modelle)

In [ ]:
results_s1 = {}
for name, cfg in MODELS.items():
    print(f'{name:4s} läuft…', end=' ', flush=True)
    yt, yp = loo_eval(X_all.values, y_coarse, cfg)
    acc = accuracy_score(yt, yp)
    results_s1[name] = {'yt': yt, 'yp': yp, 'acc': acc}
    print(f'{acc:.1%}')

print()
print('Stufe 1 — LOO-Accuracy:')
for name, r in results_s1.items():
    print(f'  {name:4s}: {r["acc"]:.1%}')

---
## 7. Stufe 2 — Feinklassifikation  (alle Modelle)

In [ ]:
results_s2 = {}
for name, cfg in MODELS.items():
    print(f'{name:4s} läuft…', end=' ', flush=True)
    yt, yp = loo_eval(X_eat.values, y_eat, cfg)
    acc = accuracy_score(yt, yp)
    results_s2[name] = {'yt': yt, 'yp': yp, 'acc': acc}
    print(f'{acc:.1%}')

print()
print('Stufe 2 — LOO-Accuracy:')
for name, r in results_s2.items():
    print(f'  {name:4s}: {r["acc"]:.1%}')

---
## 8. End-to-End LOO  (alle Modelle)

Stufe 1 und Stufe 2 werden pro Fold je neu trainiert.

In [ ]:
def e2e_loo(cfg):
    le1 = LabelEncoder().fit(y_coarse) if cfg['le'] else None
    le2 = LabelEncoder().fit(y_eat)    if cfg['le'] else None
    yt_all, yp_all = [], []
    n = len(X_all)
    for te_idx in range(n):
        tr = np.ones(n, dtype=bool); tr[te_idx] = False

        # Stufe 1
        Xtr1, Xte1 = X_all.values[tr], X_all.values[[te_idx]]
        if cfg['scale']:
            sc1  = StandardScaler()
            Xtr1 = sc1.fit_transform(Xtr1)
            Xte1 = sc1.transform(Xte1)
        ytr1 = le1.transform(y_coarse[tr]) if le1 else y_coarse[tr]
        m1   = cfg['clf']()
        m1.fit(Xtr1, ytr1)
        pred_c = m1.predict(Xte1)[0]
        if le1:
            pred_c = le1.inverse_transform([pred_c])[0]

        if pred_c == 'Essen':
            eat_tr = tr & eat_mask
            Xtr2, Xte2 = X_all.values[eat_tr], X_all.values[[te_idx]]
            if cfg['scale']:
                sc2  = StandardScaler()
                Xtr2 = sc2.fit_transform(Xtr2)
                Xte2 = sc2.transform(Xte2)
            y_eat_tr = y_raw[eat_tr]
            if len(np.unique(y_eat_tr)) < 2:
                yp_all.append(pred_c); yt_all.append(y_raw[te_idx]); continue
            le2_fold = LabelEncoder().fit(y_eat_tr) if cfg['le'] else None
            ytr2     = le2_fold.transform(y_eat_tr) if le2_fold else y_eat_tr
            m2       = cfg['clf']()
            m2.fit(Xtr2, ytr2)
            pred_f = m2.predict(Xte2)[0]
            if le2_fold:
                pred_f = le2_fold.inverse_transform([pred_f])[0]
            yp_all.append(pred_f)
        else:
            yp_all.append(pred_c)
        yt_all.append(y_raw[te_idx])
    return np.array(yt_all), np.array(yp_all)

results_e2e = {}
for name, cfg in MODELS.items():
    print(f'{name:4s} läuft…', end=' ', flush=True)
    yt, yp = e2e_loo(cfg)
    acc = accuracy_score(yt, yp)
    results_e2e[name] = {'yt': yt, 'yp': yp, 'acc': acc}
    print(f'{acc:.1%}')

print()
print('End-to-End — LOO-Accuracy:')
for name, r in results_e2e.items():
    print(f'  {name:4s}: {r["acc"]:.1%}')

---
## 9. Ergebnisübersicht

In [ ]:
print(f'{"Modell":6s}  {"Stufe 1":>9s}  {"Stufe 2":>9s}  {"End-to-End":>11s}')
print('─' * 44)
for name in MODELS:
    s1  = results_s1[name]['acc']
    s2  = results_s2[name]['acc']
    e2e = results_e2e[name]['acc']
    print(f'{name:6s}  {s1:>9.1%}  {s2:>9.1%}  {e2e:>11.1%}')
print('─' * 44)
print(f'{"NB04 RF":6s}  {NB04["S1"]:>9.1%}  {NB04["S2"]:>9.1%}  {NB04["E2E"]:>11.1%}  ← Referenz')

---
## 10. Visualisierung

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Linkes Bild: Balkendiagramm ───────────────────────────────────────────────
ax = axes[0]
names = list(MODELS.keys())
x     = np.arange(len(names))
w     = 0.25
for i, (stage_key, stage_label, res) in enumerate([
    ('S1', 'Stufe 1', results_s1),
    ('S2', 'Stufe 2', results_s2),
    ('E2E', 'End-to-End', results_e2e),
]):
    vals = [res[n]['acc'] for n in names]
    bars = ax.bar(x + (i - 1) * w, vals, w, label=stage_label,
                  alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{v:.0%}', ha='center', va='bottom', fontsize=8)

# NB04-Referenzlinie (E2E)
ax.axhline(NB04['E2E'], color='grey', linestyle='--', alpha=0.5, label='NB04 RF E2E')
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=11)
ax.set_ylabel('LOO-Accuracy'); ax.set_ylim(0.7, 1.05)
ax.set_title('Modellvergleich — LOO-CV', fontweight='bold')
ax.legend(fontsize=9)

# ── Rechtes Bild: Confusion Matrix bestes E2E-Modell ─────────────────────────
best_name = max(results_e2e, key=lambda n: results_e2e[n]['acc'])
r = results_e2e[best_name]
cm = confusion_matrix(r['yt'], r['yp'], labels=CLASSES_FULL)
ax2 = axes[1]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES_FULL, yticklabels=CLASSES_FULL, ax=ax2,
            linewidths=0.5, linecolor='white', annot_kws={'size': 11, 'weight': 'bold'}, vmin=0)
ax2.set_title(f'Bestes Modell: {best_name}  ({r["acc"]:.1%})', fontweight='bold')
ax2.set_xlabel('Vorhergesagt'); ax2.set_ylabel('Tatsächlich')

plt.suptitle('NB05 — Modellvergleich LOO-CV', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/images/nb05_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Detaillierter Report für jedes Modell (End-to-End)
for name in MODELS:
    r = results_e2e[name]
    print(f'{'═'*55}')
    print(f'  {name}  —  End-to-End  {r["acc"]:.1%}')
    print(f'{'═'*55}')
    print(classification_report(r['yt'], r['yp'], labels=CLASSES_FULL, zero_division=0))

---
## 11. Bonus: MLP mit kaskadierten Stage-1-Features

Stage-1-Ausgabe `[P(Still), P(Essen)]` wird als zusätzliche Features in Stage 2 eingespeist:

```
MLP S1: 36 Features → [64,32] → P(Still) | P(Essen)
                                     │
MLP S2: 36 Features + P(S1) → [128,64] → Apfel/Kaugummi/Skyr/Essen
```

Die S1-Probas werden via LOO berechnet — kein Data-Leakage.

In [ ]:
mlp_cfg = MODELS['MLP']

# Stage 1 LOO → Out-of-Fold Probas speichern
le_c = LabelEncoder().fit(y_coarse)
proba_s1_oof = np.zeros((len(X_all), 2))

print(f'MLP Stage 1 LOO läuft ({len(X_all)} Folds)…')
for tr, te in LeaveOneOut().split(X_all.values):
    sc   = StandardScaler()
    Xtr  = sc.fit_transform(X_all.values[tr])
    Xte  = sc.transform(X_all.values[te])
    clf  = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                          solver='adam', max_iter=500,
                          learning_rate_init=0.001, random_state=42)
    clf.fit(Xtr, le_c.transform(y_coarse[tr]))
    p = clf.predict_proba(Xte)[0]
    # Align: clf.classes_ enthält integer codes → auf [0=Essen, 1=Still] ausrichten
    for i, code in enumerate(clf.classes_):
        proba_s1_oof[te[0], code] = p[i]

print('Fertig.')

In [ ]:
# Stage 2: augmentierte Features (36 + 2 S1-Probas)
X_eat_aug = np.hstack([X_all.values[eat_mask], proba_s1_oof[eat_mask]])  # (n_eat, 38)

print(f'Stage 2 MLP-kaskadiert LOO läuft ({eat_mask.sum()} Folds)…')
le_f = LabelEncoder().fit(y_eat)
yt2_cas, yp2_cas = [], []
for tr, te in LeaveOneOut().split(X_eat_aug):
    sc   = StandardScaler()
    Xtr  = sc.fit_transform(X_eat_aug[tr])
    Xte  = sc.transform(X_eat_aug[te])
    clf  = MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu',
                          solver='adam', max_iter=500,
                          learning_rate_init=0.001, random_state=42)
    clf.fit(Xtr, le_f.transform(y_eat[tr]))
    pred = le_f.inverse_transform([clf.predict(Xte)[0]])[0]
    yp2_cas.append(pred)
    yt2_cas.append(y_eat[te[0]])

yt2_cas = np.array(yt2_cas)
yp2_cas = np.array(yp2_cas)
acc2_cas = accuracy_score(yt2_cas, yp2_cas)
print(f'\nMLP Stufe 2 kaskadiert: {acc2_cas:.1%}')
print(f'MLP Stufe 2 standard:   {results_s2["MLP"]["acc"]:.1%}')
print(f'Δ durch S1-Features:    {acc2_cas - results_s2["MLP"]["acc"]:+.1%}')

In [ ]:
print('╔══════════════════════════════════════════════════════════╗')
print('║      Finale Übersicht — alle Modelle + MLP kaskadiert    ║')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║ {"Modell":18s} {"Stufe 1":>8s}  {"Stufe 2":>8s}  {"End-to-End":>10s} ║')
print('╠══════════════════════════════════════════════════════════╣')
for name in MODELS:
    s1  = results_s1[name]['acc']
    s2  = results_s2[name]['acc']
    e2e = results_e2e[name]['acc']
    print(f'║ {name:18s} {s1:>8.1%}  {s2:>8.1%}  {e2e:>10.1%} ║')
print('╠══════════════════════════════════════════════════════════╣')
mlp_s1 = results_s1['MLP']['acc']
mlp_e2e = results_e2e['MLP']['acc']
print(f'║ {"MLP kaskadiert":18s} {mlp_s1:>8.1%}  {acc2_cas:>8.1%}  {"(—)":>10s} ║')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║ {"NB04 RF (Referenz)":18s} {NB04["S1"]:>8.1%}  {NB04["S2"]:>8.1%}  {NB04["E2E"]:>10.1%} ║')
print('╚══════════════════════════════════════════════════════════╝')